# Runtime-ordering training — can a runtime-conditioned trigger be learned?

Read `research/campaigns/runtime_h2o_validity/PREREG_runtime_training_2026_11_20.md`
first: it fixes G1–G4 (learning, full-cache dormancy, policy selectivity, ordering
specificity) before this run existed.

Order: smoke (2 examples) → build 300 long_512/long_1024 examples → train 3 epochs
with dev selection on 12 long_512 prompts → eval on 40 held-out long_512 prompts
disjoint from those 12. Every long phase has a wall-clock cap and per-epoch
checkpointing. The bins restriction and the precomputed selection sets are recorded
in the prereg addendum (v1 was cancelled; its build had sampled only short prompts).


In [ ]:
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

REPO = "https://github.com/kartikbindra/btp-backdoor-research"
COMMIT = "d3e3d52498cb57627ed3a3614f0abcbe626ca044"
WORK = Path("/kaggle/working")
SRC = Path("/kaggle/tmp/btp-research")   # outside /kaggle/working: only results become output
OUT = WORK / "rt_train_s42"

gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,uuid,driver_version,compute_cap", "--format=csv,noheader"],
    text=True).strip()
print(gpu); print(sys.version); print(platform.platform())
assert "Tesla" in gpu or "NVIDIA" in gpu

SRC.parent.mkdir(parents=True, exist_ok=True)
if SRC.exists():
    shutil.rmtree(SRC)
subprocess.run(["git", "clone", "--quiet", REPO, str(SRC)], check=True)
subprocess.run(["git", "-C", str(SRC), "checkout", "--quiet", COMMIT], check=True)
head = subprocess.check_output(["git", "-C", str(SRC), "rev-parse", "HEAD"], text=True).strip()
print("repo head", head)
for rel in ("src/pfseb/runtime_train.py", "scripts/run_runtime_train.py",
            "research/campaigns/runtime_h2o_validity/PREREG_runtime_training_2026_11_20.md"):
    p = SRC / rel
    assert p.exists(), f"missing {rel} at {head}"
    print("ok", rel)

import torch, transformers
print("torch", torch.__version__, "transformers", transformers.__version__,
      "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0))
OUT.mkdir(parents=True, exist_ok=True)
(WORK / "session_env.json").write_text(json.dumps({
    "gpu": gpu, "python": sys.version, "platform": platform.platform(),
    "torch": torch.__version__, "transformers": transformers.__version__, "repo_head": head,
}, indent=2))


In [ ]:
def run_phase(tag, args, out_dir=None, max_minutes=0):
    target = out_dir or str(OUT)
    cmd = [sys.executable, "-u", "-m", "scripts.run_runtime_train", "--out_dir", target] + args
    if max_minutes:
        cmd += ["--max_minutes", str(max_minutes)]
    print("RUN", " ".join(cmd), flush=True)
    started = time.time()
    with (WORK / f"{tag}.log").open("w", buffering=1) as log:
        proc = subprocess.run(cmd, cwd=str(SRC), stdout=log, stderr=subprocess.STDOUT)
    print(tag, "exit", proc.returncode, "minutes", round((time.time() - started) / 60, 1), flush=True)
    return proc.returncode


In [ ]:
# Smoke: a real 1.5B step on the runtime-ordering loss, a selection pass, and one eval prompt.
# Catches a broken 4-D mask / checkpointing interaction on this image before the long build.
SMOKE = WORK / "smoke"
SMOKE.mkdir(exist_ok=True)
rc1 = run_phase("smoke_build", ["--phase", "build", "--n_train", "2", "--bins", "long_512"],
                out_dir=str(SMOKE))
rc2 = run_phase("smoke_train", ["--phase", "train", "--epochs", "1", "--select_n", "1",
                                "--select_tokens", "4", "--bins", "long_512"], out_dir=str(SMOKE))
rc3 = run_phase("smoke_eval", ["--phase", "eval", "--bins", "long_512", "--max_items", "1",
                               "--max_new_tokens", "4"], out_dir=str(SMOKE))
PROCEED = (rc1 == 0 and rc2 == 0 and rc3 == 0 and (SMOKE / "theta_b.pt").exists())
print("smoke passed:", PROCEED)


In [ ]:
if PROCEED:
    run_phase("build", ["--phase", "build", "--n_train", "300",
                        "--bins", "long_512,long_1024"], max_minutes=40)
else:
    print("skipped: smoke failed")


In [ ]:
if PROCEED and (OUT / "examples.pt").exists():
    run_phase("train", ["--phase", "train", "--epochs", "3", "--select_n", "12",
                        "--select_tokens", "8", "--bins", "long_512"], max_minutes=200)
else:
    print("skipped: no examples")


In [ ]:
# G1-G4 are read off this run: 40 held-out dev prompts, disjoint from the 30 selection prompts.
if (OUT / "theta_b.pt").exists():
    run_phase("eval", ["--phase", "eval", "--pool", "dev", "--bins", "long_512",
                       "--skip_items", "12", "--max_items", "40",
                       "--max_new_tokens", "40"], max_minutes=110)
    print((OUT / "summary.json").read_text()[:2500])
else:
    print("no adapter to evaluate")


In [ ]:
stamp = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
bundle = WORK / f"runtime_h2o_train_{stamp}"
bundle.mkdir(exist_ok=True)
for root in (OUT, WORK / "smoke"):
    if root.exists():
        shutil.copytree(root, bundle / root.name, dirs_exist_ok=True)
for item in WORK.iterdir():
    if item.is_file() and (item.suffix in {".json", ".log"} or item.name.endswith(".jsonl")):
        shutil.copy2(item, bundle / item.name)
shutil.make_archive(str(WORK / "runtime_h2o_train_outputs"), "zip", bundle)
print("packaged", bundle)
print(sorted(p.name for p in bundle.iterdir()))
